# 生成策略

<b>在大语言模型（LLM）的推理过程中，“生成策略”决定了模型如何从概率分布中选择下一个词（Token）。</b>

<font color='red'>简单来说，模型每一步都会输出一个包含数万个概率值的列表（对应词表中的每个词），生成策略就是决定“选哪个词”的规则。</font>

以下是目前主流的生成策略，通常分为确定性策略和随机性策略两大类：


## 🎯 确定性策略

这类策略每次运行都会得到相同的结果，适合需要标准答案的场景（如数学计算、代码生成）。

### 贪心解码

这是最基础、最简单的策略。

- 原理：在每一步，只选择概率最高的那个词。
- 特点：
    - 速度快，不需要采样。
    - 结果完全确定。
    - 缺点：容易陷入死循环（重复生成相同的词），且生成的文本往往比较枯燥、缺乏多样性。
- 对应参数：temperature=0。
### 束搜索

贪心解码只看眼前（局部最优），而束搜索会“多看几步”（全局最优）。

- 原理：
    - 每一步<font color='red'>不只保留概率最高的一个词，而是保留概率最高的 k 个</font>（比如 k=5）候选序列。
    - 继续生成下一步，计算这 5 个序列的总概率，再选出总概率最高的 5 个。
    - 最后从所有完成的序列中选出得分最高的一个。
- 特点：
    - 通常用于机器翻译或摘要生成，因为这类任务需要准确覆盖原文信息。
    - 缺点：计算量大（需要并行计算多个序列），且生成的文本往往过于保守，缺乏人类的自然感。
- 对应参数：num_beams=k。

## 🎲 随机性策略

这类策略引入了随机性，让模型的回答更丰富、更像人类，适合聊天、创作等场景。

### 温度采样

这是控制随机性的核心“旋钮”。

- 原理：在采样前，将概率分布除以温度值 T ，然后再进行归一化。
    - 高温 ( T>1 )：概率分布变得更平缓，低概率的词也有机会被选中（更疯狂、更多样）。
    - 低温 ( T<1 )：概率分布变得更尖锐，高概率的词更容易被选中（更严谨、更确定）。
    - T=0 ：退化为贪心解码。
- 对应参数：temperature=0.7 (常用值)。
### Top-k 采样
- 原理：
    - <font color='red'>只看概率最高的 k 个词（例如 k=50）</font>。
    - 将这 50 个词的概率重新归一化（加起来等于 1），然后从中随机采样。
    - 作用：直接切掉长尾中那些不合理的、生僻的词，防止模型“胡言乱语”。
- 对应参数：top_k=50。
### Top-p 采样 (核采样)

<font color='red'>这是目前最流行的策略，比 Top-k 更灵活。</font>

- 原理：
    - <font color='red'>不固定候选词的数量，而是固定概率的累积和。</font>
    - 从概率最高的词开始累加，直到总和超过 p（例如 p=0.9）。
    - 动态调整：
         - <font color='red'>如果模型很确定（比如概率集中在几个词上），候选列表会很短。</font>
         - <font color='red'>如果模型很犹豫（概率分布很平），候选列表会变长。</font>
- 优势：比 Top-k 更能适应不同的上下文语境。
- 对应参数：top_p=0.9 (常用值)。

## 🛡️ 约束与惩罚策略

为了防止生成质量下降，通常会配合上述策略使用以下手段：

### 重复惩罚
- 作用：如果某个词或短语已经出现过，就降低它在下一步被选中的概率。
- 目的：防止模型像复读机一样无限循环（例如：“我不知道，我不知道，我不知道...”）。
- 对应参数：repetition_penalty=1.1。
### 最小概率阈值
- 作用：设定一个最低概率门槛，低于这个门槛的词直接剔除。
- 目的：比 Top-p 更激进地过滤掉不靠谱的词。
- 对应参数：min_p=0.05。
### 长度惩罚
- 作用：根据序列的长度调整分数。
- 目的：防止模型生成太短（没说完）或太长（啰嗦）的内容。

| 策略 | 核心逻辑 | 适用场景 | 优点 | 缺点 |
| :--- | :--- | :--- | :--- | :--- |
| 贪心解码 | 选概率最大的 | 事实问答、代码 | 稳定、快速 | 枯燥、易重复 |
| 束搜索 | 选全局最优路径 | 翻译、摘要 | 准确度高 | 慢、缺乏多样性 |
| 随机采样 | 按概率随机选 | 聊天、小说创作 | 自然、丰富 | 结果不可控 |
| Top-k / Top-p | 限制候选范围 | 通用推荐 | 平衡了质量与多样性 | 需调节参数 |



## 📌 最佳实践建议

在大多数现代大模型（如 Llama 3, Qwen 2.5）的日常使用中，推荐的“黄金组合”通常是：

<font color='red'>Temperature=0.7 + Top-p=0.9 + Top-k=50 + Repetition Penalty=1.05</font>

这个组合能保证回答既聪明准确，又具有一定的自然变化。